In [1]:
import pandas as pd
from urllib.request import Request, urlopen
import re
import os
import pickle
import pdfplumber
from datetime import datetime, timedelta
import requests
from pathlib import Path

# List of FNO Stocks

In [2]:
#Run periodically to update symbols and stocks list
'''
fno_list = pd.read_csv('./Cache/fno_stocks.csv')

drop_words = [' Limited', ' Ltd', ' Industries', 'The ', ' (India)', ' (india)',' Enterprises',' Enterprise', ' Company', ' Laboratories', ' Corporation']
fno_list['Stock'] = fno_list['Stock Name']

for word in drop_words:
    fno_list['Stock'] = fno_list['Stock'].map(lambda x: x.replace(word, ''))
    
fno_list['Stock'] = fno_list['Stock'].map(lambda x: x.lower())
fno_list['Symbol'] = fno_list['Symbol'].map(lambda x: x.lower())

symbols = fno_list['Symbol'].values
stocks = fno_list['Stock'].values

with open('./Cache/symbols.pkl', 'wb') as f:
    pickle.dump(symbols, f)
with open('./Cache/stocks.pkl', 'wb') as f:
    pickle.dump(stocks, f)
'''

"\nfno_list = pd.read_csv('./Cache/fno_stocks.csv')\n\ndrop_words = [' Limited', ' Ltd', ' Industries', 'The ', ' (India)', ' (india)',' Enterprises',' Enterprise', ' Company', ' Laboratories', ' Corporation']\nfno_list['Stock'] = fno_list['Stock Name']\n\nfor word in drop_words:\n    fno_list['Stock'] = fno_list['Stock'].map(lambda x: x.replace(word, ''))\n    \nfno_list['Stock'] = fno_list['Stock'].map(lambda x: x.lower())\nfno_list['Symbol'] = fno_list['Symbol'].map(lambda x: x.lower())\n\nsymbols = fno_list['Symbol'].values\nstocks = fno_list['Stock'].values\n\nwith open('./Cache/symbols.pkl', 'wb') as f:\n    pickle.dump(symbols, f)\nwith open('./Cache/stocks.pkl', 'wb') as f:\n    pickle.dump(stocks, f)\n"

In [3]:
#read symbols and stocks from pickle files
with open('./Cache/symbols.pkl', 'rb') as f:
    symbols = pickle.load(f)

In [4]:
#Download CSV from NSE website

# --- Paths ---
out_dir = Path("./Data/NSE")
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / "nse_corporate_filings.csv"

# --- URLs ---
base_url = "https://www.nseindia.com"
csv_url = "https://www.nseindia.com/api/corporate-announcements?index=equities"

# --- Headers (critical for NSE) ---
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)",
    "Accept": "application/json",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.nseindia.com/companies-listing/corporate-filings-announcements",
}

# --- Session to persist cookies ---
session = requests.Session()

# Step 1: Hit NSE homepage to get cookies
session.get(base_url, headers=headers, timeout=10)

# Step 2: Download corporate announcements CSV data
resp = session.get(csv_url, headers=headers, timeout=15)
resp.raise_for_status()

# NSE returns JSON; convert to CSV-like storage
data = resp.json()

import pandas as pd

df = pd.DataFrame(data)
df.to_csv(out_file, index=False)

print(f"Saved {len(df)} rows to {out_file}")


Saved 20 rows to Data/NSE/nse_corporate_filings.csv


In [5]:
import requests
from datetime import datetime, timedelta
from pathlib import Path

# -----------------------------
# Paths
# -----------------------------
out_dir = Path("./Data/NSE")
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / "nse_corporate_filings_1D.csv"

# -----------------------------
# Dates (DD-MM-YYYY)
# -----------------------------
today = datetime.now().strftime("%d-%m-%Y")
yesterday = (datetime.now() - timedelta(days=1)).strftime("%d-%m-%Y")

# -----------------------------
# URLs
# -----------------------------
base_url = "https://www.nseindia.com"
page_url = "https://www.nseindia.com/companies-listing/corporate-filings-announcements"
api_url = "https://www.nseindia.com/api/corporate-announcements"

params = {
    "index": "equities",
    "from_date": yesterday,
    "to_date": today,
    "reqXbrl": "false",
    "csv": "true"
}

# -----------------------------
# Headers (strict)
# -----------------------------
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)",
    "Accept": "text/csv,application/json",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate",
    "Referer": page_url,
    "Connection": "keep-alive",
}

# -----------------------------
# Session
# -----------------------------
session = requests.Session()

# Warm-ups (MANDATORY)
session.get(base_url, headers=headers, timeout=10)
session.get(page_url, headers=headers, timeout=10)

# -----------------------------
# CSV Download
# -----------------------------
resp = session.get(api_url, headers=headers, params=params, timeout=20)
resp.raise_for_status()

# Save raw CSV
out_file.write_bytes(resp.content)

print(f"Saved CSV to {out_file}")


Saved CSV to Data/NSE/nse_corporate_filings_1D.csv


# Read Announcements from donwloaded CSV

In [6]:
# Function: load and filter NSE announcements by FNO symbols and date
def load_filter_nse(nse_csv, symbols):
    df = pd.read_csv(nse_csv)
    df['SYMBOL'] = df['SYMBOL'].apply(lambda x: str(x).lower())

    df = df[df['SYMBOL'].isin(symbols)].reset_index(drop=True)

    important_subject = ['Reply to Clarification Sought', 'Press Release', 'Updates', 'News Verification',
                        'Memorandum of Understanding/Agreements','Acquisition-XBRL','Acquisition', 'Amalgamation OR Merger-XBRL',
                        'ISD for Buyback-Tender Offer', 'Amalgamation/Merger']
    
    nse_imp = df[df['SUBJECT'].isin(important_subject)].copy()
    nse_imp = nse_imp.reset_index(drop=True)

    return nse_imp

In [7]:
nse_imp = load_filter_nse('./Data/NSE/nse_corporate_filings_1D.csv', symbols)

In [8]:
nse_imp

,SYMBOL,COMPANY NAME,SUBJECT,DETAILS,BROADCAST DATE/TIME,RECEIPT,DISSEMINATION,DIFFERENCE,ATTACHMENT
0,wipro,Wipro Limited,Updates,Wipro Limited has informed the Exchange regard...,07-Jan-2026 20:39:41,2026-01-07 20:39:41,07-Jan-2026 20:39:42,00:00:01,https://nsearchives.nseindia.com/corporate/Csl...
1,canbk,Canara Bank,Updates,Canara Bank has informed the Exchange regardin...,07-Jan-2026 19:56:39,2026-01-07 19:56:39,07-Jan-2026 19:56:40,00:00:01,https://nsearchives.nseindia.com/corporate/CAN...
2,vedl,Vedanta Limited,Updates,Vedanta Limited has informed the Exchange rega...,07-Jan-2026 18:44:48,2026-01-07 18:44:48,07-Jan-2026 18:44:49,00:00:01,https://nsearchives.nseindia.com/corporate/VED...
3,bankindia,Bank of India,Updates,Bank Of India has informed the Exchange regard...,07-Jan-2026 18:42:29,2026-01-07 18:42:29,07-Jan-2026 18:42:30,00:00:01,https://nsearchives.nseindia.com/corporate/BAN...
4,cipla,Cipla Limited,News Verification,The Exchange has sought clarification from Cip...,07-Jan-2026 18:15:35,2026-01-07 18:15:35,07-Jan-2026 18:15:36,00:00:01,https://nsearchives.nseindia.com/corporate/CIP...
5,hudco,Housing & Urban Development Corporation Limited,Updates,Housing & Urban Development Corporation Limite...,07-Jan-2026 17:52:55,2026-01-07 17:52:55,07-Jan-2026 17:52:55,00:00:00,https://nsearchives.nseindia.com/corporate/HUD...
6,tatasteel,Tata Steel Limited,Press Release,Tata Steel Limited has informed the Exchange r...,07-Jan-2026 17:46:51,2026-01-07 17:46:51,07-Jan-2026 17:46:52,00:00:01,https://nsearchives.nseindia.com/corporate/NID...
7,ongc,Oil & Natural Gas Corporation Limited,Press Release,Oil & Natural Gas Corporation Limited has info...,07-Jan-2026 17:45:43,2026-01-07 17:45:43,07-Jan-2026 17:45:44,00:00:01,https://nsearchives.nseindia.com/corporate/ONG...
8,adanient,Adani Enterprises Limited,Updates,Adani Enterprises Limited has informed the Exc...,07-Jan-2026 17:40:50,2026-01-07 17:40:50,07-Jan-2026 17:40:51,00:00:01,https://nsearchives.nseindia.com/corporate/ADA...
9,boschltd,Bosch Limited,Updates,Bosch Limited has informed the Exchange regard...,07-Jan-2026 17:31:01,2026-01-07 17:31:01,07-Jan-2026 17:31:03,00:00:02,https://nsearchives.nseindia.com/corporate/BOS...


In [9]:
def extract_pdf_text(pdf_path):
    text_pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                text_pages.append(text)
    return "\n".join(text_pages)


In [10]:
def download_pdf(url, out_path):
    headers = {
        "User-Agent": "Mozilla/5.0",
    }
    r = requests.get(url, headers=headers, timeout=15)
    r.raise_for_status()

    with open(out_path, "wb") as f:
        f.write(r.content)

In [11]:
BOILERPLATE_PATTERNS = [
    r"to,\s*bse limited",
    r"to,\s*the manager",
    r"kind attention",
    r"dear sir|dear madam",
    r"yours sincerely",
    r"thank you",
    r"sd/-",
    r"registered office",
    r"corporate office",
    r"cin[:\s]",
    r"this is for your information",
]

def clean_pdf_text(text):
    lines = text.split("\n")
    cleaned = []

    for line in lines:
        l = line.strip().lower()
        if len(l) < 5:
            continue
        if any(re.search(p, l) for p in BOILERPLATE_PATTERNS):
            continue
        cleaned.append(line)

    return "\n".join(cleaned)


In [12]:
def extract_event_body(text, min_words=50):
    paragraphs = text.split("\n\n")
    paragraphs = [p.strip() for p in paragraphs if len(p.split()) > min_words]
    return "\n".join(paragraphs)

In [13]:
def process_nse_imp_extract_events(nse_imp, attachment_col='ATTACHMENT', out_col='NEWS_EVENT', error_col='EXTRACTION_ERROR', temp_path='./Data/NSE/temp.pdf'):
    """Process every row in `nse_imp`, download the attachment PDF, extract and clean text,
    and extract the event body. Writes results to `out_col` and any error text to `error_col`.
    The function removes the temporary file after each iteration. Returns the modified DataFrame."""
    # prepare output columns
    nse_imp[out_col] = None
    nse_imp[error_col] = ''

    for i in nse_imp.index:
        link = nse_imp.loc[i, attachment_col]
        try:
            # download attachment to temporary file
            download_pdf(link, temp_path)
            # extract text from the downloaded pdf
            text = extract_pdf_text(temp_path)
            # clean and extract event body
            clean_text = clean_pdf_text(text)
            event_body = extract_event_body(clean_text)
            nse_imp.loc[i, out_col] = event_body
        except Exception as e:
            # record error and continue with next row
            nse_imp.loc[i, out_col] = None
            nse_imp.loc[i, error_col] = str(e)
        finally:
            # ensure temporary file is removed
            try:
                if os.path.exists(temp_path):
                    os.remove(temp_path)
            except Exception:
                pass

    return nse_imp

In [14]:
nse_imp = load_filter_nse('./Data/NSE/nse_corporate_filings_1D.csv', symbols)
nse_imp = process_nse_imp_extract_events(nse_imp)

Cannot set gray stroke color because /'P0' is an invalid float value
Cannot set non-stroke color because 2 components are specified but only 1 (grayscale), 3 (rgb) and 4 (cmyk) are supported
Cannot set non-stroke color because 2 components are specified but only 1 (grayscale), 3 (rgb) and 4 (cmyk) are supported
Cannot set non-stroke color because 2 components are specified but only 1 (grayscale), 3 (rgb) and 4 (cmyk) are supported
Cannot set non-stroke color because 2 components are specified but only 1 (grayscale), 3 (rgb) and 4 (cmyk) are supported
Cannot set non-stroke color because 2 components are specified but only 1 (grayscale), 3 (rgb) and 4 (cmyk) are supported
Cannot set non-stroke color because 2 components are specified but only 1 (grayscale), 3 (rgb) and 4 (cmyk) are supported
Cannot set non-stroke color because 2 components are specified but only 1 (grayscale), 3 (rgb) and 4 (cmyk) are supported
Cannot set non-stroke color because 2 components are specified but only 1 (gr

In [15]:
#Exporting
nse_imp.to_pickle("./Data/NSE/nse_news.pkl")